<a href="https://colab.research.google.com/github/Hugo-Zh0/YoloV12-Object-Detection-Project/blob/main/YOLOv12_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🚀 YOLOv12 Object Detection Project

[![Python](https://img.shields.io/badge/Python-3.10%2B-blue.svg)](https://www.python.org/downloads/)
[![Anaconda](https://img.shields.io/badge/Anaconda-Navigator-green.svg)](https://www.anaconda.com/download)
[![VS Code](https://img.shields.io/badge/Editor-VS%20Code-blue.svg)](https://code.visualstudio.com/)
[![Ultralytics](https://img.shields.io/badge/YOLOv12-Ultralytics-yellow.svg)](https://github.com/ultralytics/ultralytics)
[![License](https://img.shields.io/badge/License-MIT-lightgrey.svg)](LICENSE)

---
<br>

## 📌 Overview
A **collaborative group project** by Swinburne University students in partnership with **CSIRO**.  
This repository contains the setup, configuration, and workflow for training and running **YOLOv12** object detection models.

**👨‍💻 Team Members:** Harron, Feng, Bunmi, Huss, Hugo.

---
**Repository:** `YoloV12-Object-Detection-Project`  
**What you’ll do:**
1. Check runtime & GPU (python, cuda, pytorch)
2. Install dependencies (python, ultralytics, onnx, torch, cv2)
3. Clone Github Repository (use scripts to further setup folders, paths and importing datasets)
5. Train (set your configurations and follow steps in the code)
6. Validate (validates the trained model to get metrics evaluation)
7. Predict (set your configuration and follows steps in the code to inference on your trained model using images and videos)
8. Comparison & Selection (follow script to compare all trained models and select best model to be used for next training)
9. Expor (run script to export your final trained model as an ONNX file for deployment)
10. Troubleshooting Steps

<br>

#### **Full Process Run Through**

Process 1: Train > Validate > Inference > (repeat steps) to get multiple models with different results

Processs 2: Comparison & Selection (gets best model) > use best model > repeat Process 1

Process 3: Comparison & Select (gets best model) > use best model > export model to ONNX File

<br>

#### **What to do after completion or if you don't want to run anymore**

After completing this colab you will need to export the folders(step 9) which includes the repository and runs, as the runtime session will expire when you close the website **(meaning the folders gets deleted)**.

<br>

#### **Starting from previous session**

If you are rerunning this agin, you will need to manually upload the folders back into the google colab again (it has to be zipped up first to be uploaded) Or using the google drive mount script, and the unzip repository script.
Then you will need to run script to extract the folders back to original state.

From there you can start from running the scripts in order (skipping steps that you have already done previously), then start your train, validation, inference, comparison & selection.

<br>

#### **Final Export for deployment**

After completing this colab you will need to export the folders(step 9) which is to run the script for Model Exportation. (you will need to configure the settings to your final Model PT File) Finally it will export as an **ONNX File**

# **Step 1 — Runtime & GPU check**

## **Prerequistes**
*   Change runtime type to T4-GPU
*   Change runtime to Python 3
*   Have your dataset already downloaded







In [ ]:
#@title Check Python, CUDA, and PyTorch (Checks if runtime is all correct)

import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device:", torch.cuda.get_device_name(0))
except Exception as e:
    print("PyTorch not installed yet (will install in next step).")

# Step 2 — Install dependencies

These are the python libraries and Ultralytics libraries needed to run the framework

In [ ]:
#@title Install Ultralytics & helpers

!pip install -q ultralytics
!pip install -q onnx onnxslim onnxruntime-gpu

import torch, cv2, ultralytics, onnx, onnxruntime

print("Ultralytics:", ultralytics.__version__)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("OpenCV:", cv2.__version__)
print("ONNX:", onnx.__version__)
print("ONNX Runtime:", onnxruntime.__version__)


# Step 3 — Clone your repository

Grabs our repository which contains our folder structure/files and folders to get started with our training

**(if you already have a saved repository in google drive you can skip cloning and use the Import Via Google Drive Script)**

In [ ]:
#@title Clone repository from github & allow user to upload dataset zip file (use this if you are running it for the first time)

REPO_URL = "https://github.com/Hugo-Zh0/YoloV12-Object-Detection-Project"
REPO_DIR = "/content/YoloV12-Object-Detection-Project"

import shutil, os
from google.colab import files

# Remove existing repo directory if it exists
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

# Clone the repo
!git clone -q {REPO_URL} {REPO_DIR}
print("Cloned into:", REPO_DIR)

# Prompt user to upload dataset ZIP file
print("lease upload your dataset ZIP file. It will be saved to /content")
uploaded = files.upload()

# Save uploaded ZIP file to /content
for filename in uploaded.keys():
    dest_path = os.path.join('/content', filename)
    print(f"Uploaded file saved as: {dest_path}")

In [ ]:
#@title Import via Google Drive (import your repository) or (dataset) if you have not already trained before

from google.colab import drive
import shutil
import os

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Step 2: Define source and destination paths
source_path = '/content/drive/MyDrive/YoloV12-Object-Detection-Project.zip'  # Update as needed
destination_path = '/content'

# Step 3: Copy from Drive to Colab root
if os.path.isdir(source_path):
    shutil.copytree(source_path, destination_path)
    print(f"Folder copied to {destination_path}")
elif os.path.isfile(source_path):
    shutil.copy2(source_path, destination_path)
    print(f"File copied to {destination_path}")
else:
    print("Source path not found. Please check the path and try again.")

# Step 4: Unmount Google Drive and clean up
drive.flush_and_unmount()
shutil.rmtree('/content/drive', ignore_errors=True)
print("Google Drive unmounted and cleaned up.")

In [4]:
#@title Unzip Respository (if imported locally or via google drive)
!unzip -q /content/YoloV12-Object-Detection-Project.zip -d /content/YoloV12-Object-Detection-Project

In [ ]:
#@title Use this if it did not extract properly or indented extraction
import os, shutil

outer = "/content/YoloV12-Object-Detection-Project"
inner = os.path.join(outer, "YoloV12-Object-Detection-Project")

for item in os.listdir(inner):
    shutil.move(os.path.join(inner, item), outer)

os.rmdir(inner)

print("Fixed: inner folder contents moved up into outer folder")


## 3.1 - Creating folders

**(Skip this step if you already have imported a previous repository from google drive)** other run scripts below to create necessary folders and files

Datasets Folder: Contains our datasets from roboflow


Yaml Folder: Contains our data.yaml file from our dataset


In [ ]:
#@title Creates two folders called "datasets & yaml" in the repository
import os
target_directory = '/content/YoloV12-Object-Detection-Project'

folders_to_create = ['datasets', 'yaml']

for folder_name in folders_to_create:
    folder_path = os.path.join(target_directory, folder_name)
    os.makedirs(folder_path, exist_ok=True)
    print(f"Created folder: {folder_path}")

## 3.2 Import & Unzipping Datasets

You should already have your datasets uploaded into the google colab. Via cloning the repo with uploading steps or through Google Drive.

Next we will need to run the scripts below to unzip and move them into the correct directory

(note - If you are importing your anatomy features dataset, you can import normally into as well via Colab by Right-Click and selecting Upload)

In [ ]:
#@title Unzip Koala Dataset
!unzip -q /content/koala.zip -d /content/YoloV12-Object-Detection-Project/datasets/koala

In [ ]:
!unzip -q /content/koala-anatomy-features.zip -d /content/YoloV12-Object-Detection-Project/datasets/koala-anatomy-features

In [ ]:
#@title Unzip Kangaroo Dataset
!unzip -q /content/kangaroo.zip -d /content/YoloV12-Object-Detection-Project/datasets/kangaroo

In [6]:
!unzip -q /content/kangaroo-anatomy-features.zip -d /content/YoloV12-Object-Detection-Project/datasets/kangaroo-anatomy-features

# Step 4 — Set model & data paths

**Step 4.1 Updating YAML Files Location**

Moving the yaml file to the YAML Folder for both base and features.

(note - for features yaml file rename to this "koala-anatomy-features.yaml" applies to kangaroo as well)

In [ ]:
#@title Yaml for Koala (Base)
import shutil
import os

source_path = '/content/YoloV12-Object-Detection-Project/datasets/koala/data.yaml'
destination_dir = '/content/YoloV12-Object-Detection-Project/yaml'


os.makedirs(destination_dir, exist_ok=True)

destination_path = os.path.join(destination_dir, os.path.basename(source_path))
shutil.move(source_path, destination_path)

print(f"File moved to: {destination_path}")

In [ ]:
#@title Yaml for Koala (Features)
import shutil
import os

source_path = '/content/YoloV12-Object-Detection-Project/datasets/koala/koala-anatomy-features.yaml'
destination_dir = '/content/YoloV12-Object-Detection-Project/yaml'


os.makedirs(destination_dir, exist_ok=True)

destination_path = os.path.join(destination_dir, os.path.basename(source_path))
shutil.move(source_path, destination_path)

print(f"File moved to: {destination_path}")

In [ ]:
#@title Yaml for Kangaroo (Base)
import shutil
import os

source_path = '/content/YoloV12-Object-Detection-Project/datasets/kangaroo/kangaroo.yaml'
destination_dir = '/content/YoloV12-Object-Detection-Project/yaml'


os.makedirs(destination_dir, exist_ok=True)

destination_path = os.path.join(destination_dir, os.path.basename(source_path))
shutil.move(source_path, destination_path)

print(f"File moved to: {destination_path}")

In [ ]:
#@title Yaml for Kangaroo (Features)
import shutil
import os

source_path = '/content/YoloV12-Object-Detection-Project/datasets/kangaroo-anatomy-features/kangaroo-anatomy-features.yaml'
destination_dir = '/content/YoloV12-Object-Detection-Project/yaml'


os.makedirs(destination_dir, exist_ok=True)

destination_path = os.path.join(destination_dir, os.path.basename(source_path))
shutil.move(source_path, destination_path)

print(f"File moved to: {destination_path}")

**Step 4.2**

Update the YAML File with proper location paths for: train, val, test

Double click the yaml file and it will open on the side.

**Koala:**

* /content/YoloV12-Object-Detection-Project/datasets/koala/train/images
* /content/YoloV12-Object-Detection-Project/datasets/koala/valid/images
* /content/YoloV12-Object-Detection-Project/datasets/koala/test/images

**Koala-Features:**

* /content/YoloV12-Object-Detection-Project/datasets/koala-anatomy-features/train/images
* /content/YoloV12-Object-Detection-Project/datasets/koala-anatomy-features/valid/images
* /content/YoloV12-Object-Detection-Project/datasets/koala-anatomy-features/test/images

<br>

**Kangaroo:**

* /content/YoloV12-Object-Detection-Project/datasets/kangaroo/train/images
* /content/YoloV12-Object-Detection-Project/datasets/kangaroo/valid/images
* /content/YoloV12-Object-Detection-Project/datasets/kangaroo/test/images


**Kangaroo-Features:**

* /content/YoloV12-Object-Detection-Project/datasets/kangaroo-anatomy-features/train/images
* /content/YoloV12-Object-Detection-Project/datasets/kangaroo-anatomy-features/valid/images
* /content/YoloV12-Object-Detection-Project/datasets/kangaroo-anatomy-features/test/images
<br>

Finally save the file doing Ctrl+S

(note - This applies to the anatomy features dataset as well)

# Step 5 - Training (set your parameters)

This script will train your dataset and store them in the respoitory locations.
You can also set the configs you want to train your dataset, tune it however you like to get the best performance.

In [ ]:
#@title Set Configs and train datasets for Koala
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from ultralytics import YOLO
import torch, os, glob

# =========================
# TOGGLES
# =========================
DATASET = "features"   # "base" or "features" (start with base to get base model, then move onto features)
RESUME  = False        # True to resume a paused FEATURES or BASE run
# =========================

ROOT = "/content/YoloV12-Object-Detection-Project"

if DATASET == "base":
    DATA_YAML     = f"{ROOT}/yaml/data.yaml"
    RUNS_DIR      = f"{ROOT}/runs/completed-training"
    START_WEIGHTS = f"{ROOT}/models/train2_best.pt"  # <- your last base best model (if none use yolo12s.pt)
elif DATASET == "features":
    DATA_YAML     = f"{ROOT}/yaml/koala-anatomy-features.yaml"
    RUNS_DIR      = f"{ROOT}/runs/completed-training/koala-anatomy-features"
    START_WEIGHTS = f"{ROOT}/models/features_train3_best.pt"  # <- your selected best model after comparison script
    # After 1st traing switch to latest train run model to use "/runs/completed-training/koala-anatomy-features/trainxxx/weights/best.pt"

else:
    raise ValueError("DATASET must be 'base' or 'features'")

print("Dataset   :", DATASET)
print("Data yaml :", DATA_YAML)
print("Runs dir  :", RUNS_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Determine whether to RESUME or start fresh-from START_WEIGHTS
resume_flag = False
weights_for_load = START_WEIGHTS

if RESUME:
    # try to pick latest run in this dataset folder and use its last.pt
    if os.path.isdir(RUNS_DIR):
        existing_runs = sorted(glob.glob(os.path.join(RUNS_DIR, "train*")), key=os.path.getmtime)
        if existing_runs:
            latest_run = existing_runs[-1]
            last_ckpt  = os.path.join(latest_run, "weights", "last.pt")
            if os.path.exists(last_ckpt):
                weights_for_load = last_ckpt
                resume_flag = True
                print(f"Resuming latest run: {latest_run}")
                print(f"Resume checkpoint   : {last_ckpt}")
            else:
                print("No last.pt found in latest run; starting new fine-tune from START_WEIGHTS.")
        else:
            print("No prior runs found; starting new fine-tune from START_WEIGHTS.")
    else:
        print("Runs folder doesn't exist yet; starting new fine-tune from START_WEIGHTS.")

print("Loading weights:", weights_for_load)
model = YOLO(weights_for_load)

EPOCHS  = 5 if DATASET == "features" else 200
IMGSZ   = 512
WORKERS = 6  # change so it works for the GPU/RAM Variant on Colab

# --- Inline augmentation overrides only for 'features' ---
features_aug = {
    # Color hues for the images
    "hsv_h": 0.30,
    "hsv_s": 0.85,
    "hsv_v": 0.55,

    # Geometric distorts the image perspective into different angles and size
    "degrees": 15.0,
    "translate": 0.25,
    "scale": 0.70,
    "shear": 2.5,
    "perspective": 0.0010,

    # Flips tthe images around
    "fliplr": 0.50,
    "flipud": 0.0,

    # Merges images into one big object (mosaic setting)
    "mosaic": 1.0,
    "mixup": 0.30,
    "copy_paste": 0.0,

    # Stabilize the last 20 images after merging
    "close_mosaic": 20,
}

aug_overrides = features_aug if DATASET == "features" else {}

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=-1,
    workers=WORKERS,
    device=device,
    pretrained=False,
    amp=False,
    #multi_scale=Talse,
    label_smoothing=0.05 if DATASET == "features" else 0.0,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.1,
    cos_lr=True,
    weight_decay=0.0005,
    patience=200,
    project=RUNS_DIR,   # creates train, train2, train3... under this folder
    name="train",
    resume=resume_flag,
    **aug_overrides #trains with augmentation at the same time
)

print("Save dir:", results.save_dir)

In [ ]:
#@title Set Configs and train datasaet for Kangaroo
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from ultralytics import YOLO
import torch, os, glob

# =========================
# TOGGLES
# =========================
DATASET = "features"   # "base" or "features" (start with base to get base model, then move onto features)
RESUME  = False        # True to resume a paused FEATURES or BASE run
# =========================

ROOT = "/content/YoloV12-Object-Detection-Project"

if DATASET == "base":
    DATA_YAML     = f"{ROOT}/yaml/data.yaml"
    RUNS_DIR      = f"{ROOT}/runs/completed-training"
    START_WEIGHTS = f"{ROOT}/models/yolo12s.pt"  # <- your last base best model  (if none use yolo12s.pt)
elif DATASET == "features":
    DATA_YAML     = f"{ROOT}/yaml/kangaroo-anatomy-features.yaml"
    RUNS_DIR      = f"{ROOT}/runs/completed-training/kangaroo-anatomy-features"
    START_WEIGHTS = f"{ROOT}/models/features_train3_best.pt"  # <- your selected best model after comparison script
    # After 1st training switch to latest train run model to use "/runs/completed-training/kangaroo-anatomy-features/trainxxx/weights/best.pt"
else:
    raise ValueError("DATASET must be 'base' or 'features'")

print("Dataset   :", DATASET)
print("Data yaml :", DATA_YAML)
print("Runs dir  :", RUNS_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Determine whether to RESUME or start fresh-from START_WEIGHTS
resume_flag = False
weights_for_load = START_WEIGHTS

if RESUME:
    # try to pick latest run in this dataset folder and use its last.pt
    if os.path.isdir(RUNS_DIR):
        existing_runs = sorted(glob.glob(os.path.join(RUNS_DIR, "train*")), key=os.path.getmtime)
        if existing_runs:
            latest_run = existing_runs[-1]
            last_ckpt  = os.path.join(latest_run, "weights", "last.pt")
            if os.path.exists(last_ckpt):
                weights_for_load = last_ckpt
                resume_flag = True
                print(f"Resuming latest run: {latest_run}")
                print(f"Resume checkpoint   : {last_ckpt}")
            else:
                print("No last.pt found in latest run; starting new fine-tune from START_WEIGHTS.")
        else:
            print("No prior runs found; starting new fine-tune from START_WEIGHTS.")
    else:
        print("Runs folder doesn't exist yet; starting new fine-tune from START_WEIGHTS.")

print("Loading weights:", weights_for_load)
model = YOLO(weights_for_load)

EPOCHS  = 150 if DATASET == "features" else 150
IMGSZ   = 960
WORKERS = 6  # change so it works for the GPU/RAM Variant on Colab

# --- Inline augmentation overrides only for 'features' ---
features_aug = {
    # Color shifts minimal
    "hsv_h": 0.10,
    "hsv_s": 0.45,
    "hsv_v": 0.30,

    # Gentle geometry
    "degrees": 5.0,
    "translate": 0.10,
    "scale": 0.90,
    "shear": 0.5,
    "perspective": 0.0002,

    # Flips
    "fliplr": 0.25,
    "flipud": 0.0,

    # Mosaic and blending mostly off
    "mosaic": 0.40,
    "mixup": 0.05,
    "copy_paste": 0.0,

    # Stabilize last few batches
    "close_mosaic": 10,
}

aug_overrides = features_aug if DATASET == "features" else {}

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=-1,
    workers=WORKERS,
    device=device,
    pretrained=False,
    amp=False,
    #multi_scale=Talse,
    label_smoothing=0.05 if DATASET == "features" else 0.0,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.1,
    cos_lr=True,
    weight_decay=0.0005,
    patience=100,
    project=RUNS_DIR,   # creates train, train2, train3... under this folder
    name="train",
    resume=resume_flag,
    **aug_overrides #trains with augmentation at the same time
)

print("Save dir:", results.save_dir)

# Step 6 — Test latest trained model

This script will grab the latest trained runs, uses its best.pt to validate its correct file and do a test on the model to see if detection is happening correctly, looking for accuracy, bounding boxes and efficiency

In [ ]:
#@title Test latest trained model for Koala

import glob, os
from ultralytics import YOLO

# =========================
# TOGGLE
# =========================
DATASET = "features"   # "base" or "features"
# =========================

ROOT = "/content/YoloV12-Object-Detection-Project"

if DATASET == "base":
    RUNS_DIR  = f"{ROOT}/runs/completed-training"
    DATA_YAML = f"{ROOT}/yaml/data.yaml"
elif DATASET == "features":
    RUNS_DIR  = f"{ROOT}/runs/completed-training/koala-anatomy-features"
    DATA_YAML = f"{ROOT}/yaml/koala-anatomy-features.yaml"
else:
    raise ValueError("DATASET must be 'base' or 'features'")

RESULTS_DIR    = f"{ROOT}/runs/test-results"
MODEL_TEST_DIR = f"{ROOT}/runs/test-models/validation"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_TEST_DIR, exist_ok=True)

all_runs = sorted(glob.glob(os.path.join(RUNS_DIR, "train*")))

for run_path in all_runs:
    run_name = os.path.basename(run_path)
    weights_path = os.path.join(run_path, "weights", "best.pt")
    save_path = os.path.join(RESULTS_DIR, f"{DATASET}_{run_name}_metrics.txt")  # tag with dataset

    if not os.path.exists(weights_path):
        continue
    if os.path.exists(save_path):
        continue

    print(f"[{DATASET}] Validating run: {run_name}")
    print(f"Using weights: {weights_path}")

    model = YOLO(weights_path)
    metrics = model.val(
        data=DATA_YAML,
        split="test",
        imgsz=960,
        batch=8,
        project=MODEL_TEST_DIR,
        name=f"{DATASET}_{run_name}",
        exist_ok=True
    )

    with open(save_path, "w") as f:
        f.write(f"dataset: {DATASET}\n")
        f.write(f"Validation results for run: {run_name}\n")
        f.write(f"Weights: {weights_path}\n\n")
        for k, v in metrics.results_dict.items():
            f.write(f"{k}: {v}\n")

    print(f"Metrics saved to {save_path}")

In [ ]:
#@title Test latest trained model for Kangaroo

import glob, os
from ultralytics import YOLO

# =========================
# TOGGLE
# =========================
DATASET = "features"   # "base" or "features" (selecct your options here)
# =========================

ROOT = "/content/YoloV12-Object-Detection-Project"

if DATASET == "base":
    RUNS_DIR  = f"{ROOT}/runs/completed-training"
    DATA_YAML = f"{ROOT}/yaml/data.yaml"
elif DATASET == "features":
    RUNS_DIR  = f"{ROOT}/runs/completed-training/kangaroo-anatomy-features"
    DATA_YAML = f"{ROOT}/yaml/kangaroo-anatomy-features.yaml"
else:
    raise ValueError("DATASET must be 'base' or 'features'")

RESULTS_DIR    = f"{ROOT}/runs/test-results"
MODEL_TEST_DIR = f"{ROOT}/runs/test-models/validation"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_TEST_DIR, exist_ok=True)

all_runs = sorted(glob.glob(os.path.join(RUNS_DIR, "train*")))

for run_path in all_runs:
    run_name = os.path.basename(run_path)
    weights_path = os.path.join(run_path, "weights", "best.pt") # uses the pt files in this location
    save_path = os.path.join(RESULTS_DIR, f"{DATASET}_{run_name}_metrics.txt")  # creates the txt file with dataset name

    if not os.path.exists(weights_path):
        continue
    if os.path.exists(save_path):
        continue

    print(f"[{DATASET}] Validating run: {run_name}")
    print(f"Using weights: {weights_path}")

    model = YOLO(weights_path)
    metrics = model.val(
        data=DATA_YAML,
        split="test",
        imgsz=512,
        batch=8,
        project=MODEL_TEST_DIR,
        name=f"{DATASET}_{run_name}",
        exist_ok=True
    )

    with open(save_path, "w") as f:
        f.write(f"dataset: {DATASET}\n")
        f.write(f"Validation results for run: {run_name}\n")
        f.write(f"Weights: {weights_path}\n\n")
        for k, v in metrics.results_dict.items():
            f.write(f"{k}: {v}\n")

    print(f"Metrics saved to {save_path}")

# Step 7 — Inference Testing For Images And Videos


After completeing a model test from step 6, we will move onto testing with new images and videos. Which can be sourced online anywhere. This way testing will be more accurate as its new data that the model hasn't been trained/tested on.


After running the create folder scripts for your specific dataset, you can now run upload whatever test images or videos into the folder with templates of e.g. Koala-X.png/jpg and Koala-vid-x.png/jpg.

In [ ]:
#@title Create folder to store test images and videos **(skip this step if you already have these created previously from running the script)**
# naming scheme must be (kangaroo-vid-1.mp4 or kangaroo-1.jpeg and so on)

import os
from google.colab import files
import shutil

base_dir = "/content/YoloV12-Object-Detection-Project/testing"

# Pick ONE dataset folder to upload into
#folder_name = "koala"
folder_name = "kangaroo"

target_path = os.path.join(base_dir, folder_name)
os.makedirs(target_path, exist_ok=True)
print(f"Folder created at: {target_path}")

print("Please upload your image or video files:")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, os.path.join(target_path, filename))
    print(f"Moved '{filename}' to '{target_path}'.")

print("All files uploaded and saved in your custom directory!")

In [ ]:
#@title Inference Testing Koala (base model)

import glob
import os
from ultralytics import YOLO

# Path where YOLO saves runs
RUNS_DIR = "/content/YoloV12-Object-Detection-Project/runs/completed-training"

# Uncomment the one you are not using
TEST_DIR = "/content/YoloV12-Object-Detection-Project/testing/koala"
#TEST_DIR = "/content/YoloV12-Object-Detection-Project/testing/kangaroo"

# Base path for inference results
OUTPUT_BASE = os.path.join(TEST_DIR, "inference_results_base")
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Find the next available inference folder (inference1, inference2, ...)
i = 1
while os.path.exists(os.path.join(OUTPUT_BASE, f"inference{i}")):
    i += 1
OUTPUT_DIR = os.path.join(OUTPUT_BASE, f"inference{i}")

# Find the most recent training run
latest_run = max(glob.glob(os.path.join(RUNS_DIR, "*")), key=os.path.getmtime)
weights_path = os.path.join(latest_run, "weights", "best.pt")

print(f"Running inference with weights: {weights_path}")
print(f"Testing folder: {TEST_DIR}")
print(f"Results will be saved to: {OUTPUT_DIR}")

# Load model
model = YOLO(weights_path)

# Run inference (images + videos in same folder)
results = model.predict(
    source=TEST_DIR,   # folder containing both images & videos
    imgsz=1024,         # change between 540 960 1024 or more
    conf=0.25,         # confidence higher means only detect if 0.70 or above (experiment to find the best conf)
    save=True,
    project=OUTPUT_DIR,
    name="",           # ensures results are saved directly in OUTPUT_DIR
    exist_ok=True
)

print(f"Inference complete. Results saved to: {OUTPUT_DIR}")

In [ ]:
#@title Inference for Koala Features Only or Both
# Inference Testing with Images & Videos (Koala Anatomy Features Model)
# Modes: "both" | "features_only"

import os, glob, re, cv2
from ultralytics import YOLO

# ======= SETTINGS YOU CHANGE =======
MODE         = "both"   # "both" | "features_only"
TEST_FOLDER  = "/content/YoloV12-Object-Detection-Project/testing/koala"
INFER_IMGSZ  = 512     # match training if possible (1024/1280/1536)
# ===================================

ROOT        = "/content/YoloV12-Object-Detection-Project"
RUNS_FOLDER = f"{ROOT}/runs/completed-training/koala-anatomy-features"

if MODE not in {"both", "features_only"}:
    raise SystemExit("MODE must be 'both' or 'features_only'")

# ---------- helpers ----------
def natural_key(s: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

def list_media(folder, img_exts, vid_exts):
    paths = []
    for ext in img_exts | vid_exts:
        paths.extend(glob.glob(os.path.join(folder, f"*{ext}")))
    # de-dup + natural sort
    paths = sorted(set(paths), key=lambda p: natural_key(os.path.basename(p)))
    return paths

# ---------- outputs ----------
OUT_ROOT = os.path.join(TEST_FOLDER, f"inference_results_{MODE}")
os.makedirs(OUT_ROOT, exist_ok=True)
i = 1
while os.path.exists(os.path.join(OUT_ROOT, f"inference{i}")):
    i += 1
OUT_FOLDER = os.path.join(OUT_ROOT, f"inference{i}")
os.makedirs(OUT_FOLDER, exist_ok=True)

# ---------- load weights ----------
latest_run = max(glob.glob(os.path.join(RUNS_FOLDER, "train*")), key=os.path.getmtime)
WEIGHTS = os.path.join(latest_run, "weights", "best.pt")
print(f"[Mode={MODE}] Weights={WEIGHTS}\nInput={TEST_FOLDER}\nOutput={OUT_FOLDER}")

model = YOLO(WEIGHTS)

# ---------- classes ----------
CLASS_NAMES = {i: str(n).strip().lower() for i, n in model.names.items()}
INV_BY_NAME = {v: k for k, v in CLASS_NAMES.items()}

CANONICAL_FEATURES = ["koala_ears", "koala_nose", "koala_claws"]

def is_feature_name(name: str) -> bool:
    n = name.lower()
    # accept exact or prefix (handles accidental plural/double 's', e.g., 'koala_earss')
    return any(n.startswith(c) for c in CANONICAL_FEATURES)

KOALA_NAME = "koala" if "koala" in INV_BY_NAME else None
FEATURE_NAMES = sorted({n for n in CLASS_NAMES.values() if is_feature_name(n)})

print("Model classes:", CLASS_NAMES)
print("KOALA_NAME:", KOALA_NAME, "| FEATURE_NAMES:", FEATURE_NAMES)

# ---------- thresholds / NMS ----------
CONF     = 0.20   # explicit default
IOU_NMS  = 0.50
MAX_DET  = 100

def select_dets(result):
    koalas, features = [], []
    for b in result.boxes:
        cid   = int(b.cls.item())
        score = float(b.conf.item())
        name  = CLASS_NAMES.get(cid, f"class_{cid}")
        det = {"name": name, "score": score, "xyxy": b.xyxy.cpu().numpy()[0].tolist()}
        if KOALA_NAME and name == KOALA_NAME:
            koalas.append(det)
        elif name in FEATURE_NAMES:
            features.append(det)
    # MODE logic: only two modes supported
    return (koalas + features) if MODE == "both" else features

# ---------- enumerate files explicitly ----------
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VID_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}
media = list_media(TEST_FOLDER, IMG_EXTS, VID_EXTS)

if not media:
    raise SystemExit(f"No media found in {TEST_FOLDER}")

print("\nFiles to process:")
for p in media:
    print(" -", os.path.basename(p))
print()

# ---------- process each file individually ----------
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
processed, skipped = 0, []

for path in media:
    base = os.path.basename(path)
    stem, ext = os.path.splitext(base)
    is_video = ext.lower() in VID_EXTS

    print(f"[START] {base}")

    try:
        if is_video:
            # determine FPS safely
            cap = cv2.VideoCapture(path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            cap.release()
            if fps is None or fps <= 0:
                fps = 30.0

            out_path = os.path.join(OUT_FOLDER, f"{stem}_{MODE}.mp4")
            writer = None
            frame_count = 0

            for r in model.predict(
                source=path, imgsz=INFER_IMGSZ, conf=CONF, iou=IOU_NMS,
                max_det=MAX_DET, stream=True, save=False
            ):
                frame = r.orig_img.copy()
                chosen = select_dets(r)

                # draw
                for det in chosen:
                    x1, y1, x2, y2 = map(int, det["xyxy"])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"{det['name']} {det['score']:.2f}"
                    cv2.putText(frame, label, (x1, max(20, y1 - 6)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

                if writer is None:
                    h, w = frame.shape[:2]
                    writer = cv2.VideoWriter(out_path, fourcc, float(fps), (w, h))
                writer.write(frame)
                frame_count += 1

            if writer is not None:
                writer.release()
                print(f"[DONE] {base} -> {os.path.basename(out_path)} ({frame_count} frames)")
            else:
                print(f"[WARN] {base}: no frames produced (unsupported/empty video?)")
                skipped.append(base)

        else:
            # images
            out_path = os.path.join(OUT_FOLDER, base)
            for r in model.predict(
                source=path, imgsz=INFER_IMGSZ, conf=CONF, iou=IOU_NMS,
                max_det=MAX_DET, stream=True, save=False
            ):
                frame = r.orig_img.copy()
                chosen = select_dets(r)
                for det in chosen:
                    x1, y1, x2, y2 = map(int, det["xyxy"])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"{det['name']} {det['score']:.2f}"
                    cv2.putText(frame, label, (x1, max(20, y1 - 6)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                cv2.imwrite(out_path, frame)
                print(f"[DONE] {base} -> {os.path.basename(out_path)}")

        processed += 1

    except Exception as e:
        print(f"[ERROR] {base}: {e}")
        skipped.append(base)

print(f"\nProcessed: {processed} | Skipped: {len(skipped)}")
if skipped:
    print("Skipped files:", ", ".join(skipped))

print("Output folder:", OUT_FOLDER)

In [ ]:
#@title Inference Testing Kangaroo (base model)

import glob
import os
from ultralytics import YOLO

# Path where YOLO saves runs
RUNS_DIR = "/content/YoloV12-Object-Detection-Project/runs/completed-training"

# Uncomment the one you are not using
TEST_DIR = "/content/YoloV12-Object-Detection-Project/testing/kangaroo"

# Base path for inference results
OUTPUT_BASE = os.path.join(TEST_DIR, "inference_results_base")
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Find the next available inference folder (inference1, inference2, ...)
i = 1
while os.path.exists(os.path.join(OUTPUT_BASE, f"inference{i}")):
    i += 1
OUTPUT_DIR = os.path.join(OUTPUT_BASE, f"inference{i}")

# Find the most recent training run
latest_run = max(glob.glob(os.path.join(RUNS_DIR, "*")), key=os.path.getmtime)
weights_path = os.path.join(latest_run, "weights", "best.pt")

print(f"Running inference with weights: {weights_path}")
print(f"Testing folder: {TEST_DIR}")
print(f"Results will be saved to: {OUTPUT_DIR}")

# Load model
model = YOLO(weights_path)

# Run inference (images + videos in same folder)
results = model.predict(
    source=TEST_DIR,   # folder containing both images & videos
    imgsz=512,         # change between 512 960 1024 or more
    conf=0.25,         # confidence higher means only detect if 0.70 or above (experiment to find the best conf)
    save=True,
    project=OUTPUT_DIR,
    name="",           # ensures results are saved directly in OUTPUT_DIR
    exist_ok=True
)

print(f"Inference complete. Results saved to: {OUTPUT_DIR}")

In [ ]:
#@title Inference Testing for Kangaroo Features Only or Both
# Inference Testing with Images & Videos (Kangaroo Anatomy Features Model)
# Modes: "both" | "features_only"

import os, glob, re, cv2
from ultralytics import YOLO

# ======= SETTINGS YOU CHANGE =======
MODE         = "features_only"   # "both" | "features_only"
TEST_FOLDER  = "/content/YoloV12-Object-Detection-Project/testing/kangaroo"
INFER_IMGSZ  = 960     # match training if possible (1024/1280/1536)
# ===================================

ROOT        = "/content/YoloV12-Object-Detection-Project"
RUNS_FOLDER = f"{ROOT}/runs/completed-training/kangaroo-anatomy-features"

if MODE not in {"both", "features_only"}:
    raise SystemExit("MODE must be 'both' or 'features_only'")

# ---------- helpers ----------
def natural_key(s: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

def list_media(folder, img_exts, vid_exts):
    paths = []
    for ext in img_exts | vid_exts:
        paths.extend(glob.glob(os.path.join(folder, f"*{ext}")))
    # de-dup + natural sort
    paths = sorted(set(paths), key=lambda p: natural_key(os.path.basename(p)))
    return paths

# ---------- outputs ----------
OUT_ROOT = os.path.join(TEST_FOLDER, f"inference_results_{MODE}")
os.makedirs(OUT_ROOT, exist_ok=True)
i = 1
while os.path.exists(os.path.join(OUT_ROOT, f"inference{i}")):
    i += 1
OUT_FOLDER = os.path.join(OUT_ROOT, f"inference{i}")
os.makedirs(OUT_FOLDER, exist_ok=True)

# ---------- load weights ----------
latest_run = max(glob.glob(os.path.join(RUNS_FOLDER, "train*")), key=os.path.getmtime)
WEIGHTS = os.path.join(latest_run, "weights", "best.pt")
print(f"[Mode={MODE}] Weights={WEIGHTS}\nInput={TEST_FOLDER}\nOutput={OUT_FOLDER}")

model = YOLO(WEIGHTS)

# ---------- classes (case-sensitive) ----------
CLASS_NAMES = {i: str(n).strip() for i, n in model.names.items()}
INV_BY_NAME = {v: k for k, v in CLASS_NAMES.items()}

# canonical feature names (exact as in dataset)
CANONICAL_FEATURES = ["Kangaroo_ears", "Kangaroo_nose", "Kangaroo_tail"]
KANGAROO_NAME = "Kangaroo"

KANGAROO_ID = INV_BY_NAME.get(KANGAROO_NAME, None)
FEATURE_NAMES = set(CANONICAL_FEATURES)

print("Model classes:", CLASS_NAMES)
print("KANGAROO_ID:", KANGAROO_ID, "| FEATURE_NAMES:", FEATURE_NAMES)

# ---------- thresholds / NMS ----------
CONF     = 0.20
IOU_NMS  = 0.50
MAX_DET  = 100

def select_dets(result):
    chosen = []
    for b in getattr(result, "boxes", []) or []:
        cid   = int(b.cls.item())
        score = float(b.conf.item())
        name  = CLASS_NAMES.get(cid, f"class_{cid}")

        det = {"name": name, "score": score, "xyxy": b.xyxy.cpu().numpy()[0].tolist()}

        # keep logic simple and case-sensitive
        if MODE == "both":
            if name == KANGAROO_NAME or name in FEATURE_NAMES:
                chosen.append(det)
        elif MODE == "features_only":
            if name in FEATURE_NAMES:
                chosen.append(det)
    return chosen

# ---------- enumerate files explicitly ----------
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VID_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}
media = list_media(TEST_FOLDER, IMG_EXTS, VID_EXTS)

if not media:
    raise SystemExit(f"No media found in {TEST_FOLDER}")

print("\nFiles to process:")
for p in media:
    print(" -", os.path.basename(p))
print()

# ---------- process each file individually ----------
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
processed, skipped = 0, []

for path in media:
    base = os.path.basename(path)
    stem, ext = os.path.splitext(base)
    is_video = ext.lower() in VID_EXTS

    print(f"[START] {base}")

    try:
        if is_video:
            # determine FPS safely
            cap = cv2.VideoCapture(path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            cap.release()
            if fps is None or fps <= 0:
                fps = 30.0

            out_path = os.path.join(OUT_FOLDER, f"{stem}_{MODE}.mp4")
            writer = None
            frame_count = 0

            for r in model.predict(
                source=path, imgsz=INFER_IMGSZ, conf=CONF, iou=IOU_NMS,
                max_det=MAX_DET, stream=True, save=False
            ):
                frame = r.orig_img.copy()
                chosen = select_dets(r)

                # draw
                for det in chosen:
                    x1, y1, x2, y2 = map(int, det["xyxy"])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"{det['name']} {det['score']:.2f}"
                    cv2.putText(frame, label, (x1, max(20, y1 - 6)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

                if writer is None:
                    h, w = frame.shape[:2]
                    writer = cv2.VideoWriter(out_path, fourcc, float(fps), (w, h))
                writer.write(frame)
                frame_count += 1

            if writer is not None:
                writer.release()
                print(f"[DONE] {base} -> {os.path.basename(out_path)} ({frame_count} frames)")
            else:
                print(f"[WARN] {base}: no frames produced (unsupported/empty video?)")
                skipped.append(base)

        else:
            # images
            out_path = os.path.join(OUT_FOLDER, base)
            for r in model.predict(
                source=path, imgsz=INFER_IMGSZ, conf=CONF, iou=IOU_NMS,
                max_det=MAX_DET, stream=True, save=False
            ):
                frame = r.orig_img.copy()
                chosen = select_dets(r)
                for det in chosen:
                    x1, y1, x2, y2 = map(int, det["xyxy"])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"{det['name']} {det['score']:.2f}"
                    cv2.putText(frame, label, (x1, max(20, y1 - 6)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                cv2.imwrite(out_path, frame)
                print(f"[DONE] {base} -> {os.path.basename(out_path)}")

        processed += 1

    except Exception as e:
        print(f"[ERROR] {base}: {e}")
        skipped.append(base)

print(f"\nProcessed: {processed} | Skipped: {len(skipped)}")
if skipped:
    print("Skipped files:", ", ".join(skipped))

print("Output folder:", OUT_FOLDER)

# Step 8 - Choosing Final Model for Deployment

This step will go through every single trained model/validation results, and finds/selects the best model from the metrics provided. This way with the Final Model selected will go through final inference testing then Step 9 to be exported and used in real world deployment.

In [ ]:
#@title Compares your trained models saves them into a CSV (Koala)
# Compare YOLO runs with a dataset filter: "base" | "features" | "both"
# Outputs (depending on filter):
#   runs/model-results/model_comparison_base.csv
#   runs/model-results/model_comparison_features.csv

import os, glob, pandas as pd, sys

# ====== CONFIG (can override via --dataset base|features|both) ======
DATASET_CHOICE = "features"
# ====================================================================

# Optional CLI override: --dataset base | features | both
for i, a in enumerate(sys.argv):
    if a.startswith("--dataset"):
        if "=" in a:
            DATASET_CHOICE = a.split("=", 1)[1].strip().lower()
        elif i + 1 < len(sys.argv):
            DATASET_CHOICE = sys.argv[i+1].strip().lower()

if DATASET_CHOICE not in {"base", "features", "both"}:
    raise SystemExit("Invalid --dataset. Use: base | features | both")

ROOT = "/content/YoloV12-Object-Detection-Project"
TEST_RESULTS_DIR = f"{ROOT}/runs/test-results"
RESULTS_DIR = f"{ROOT}/runs/model-results"
os.makedirs(RESULTS_DIR, exist_ok=True)

CSV_BASE = os.path.join(RESULTS_DIR, "model_comparison_base.csv")
CSV_FEAT = os.path.join(RESULTS_DIR, "model_comparison_features.csv")

def infer_dataset(dataset_field, metrics_filename, weights_str):
    """Return 'base' or 'features' even if the file lacks a dataset header."""
    ds = (dataset_field or "").strip().lower()
    if ds in {"base", "features"}:
        return ds
    name = os.path.basename(metrics_filename).lower()
    w = (weights_str or "").lower()
    # Heuristics for features
    if name.startswith("features_") or "_features" in name:
        return "features"
    if "koala-anatomy-features" in w:
        return "features"
    # Default fallback = base
    return "base"

def parse_metrics_file(path):
    dataset = ""
    run = ""
    weights = ""
    precision = recall = map50 = map5095 = fitness = None

    with open(path, "r") as f:
        for raw in f:
            ln = raw.strip()
            if ln.startswith("dataset:"):
                dataset = ln.split(":", 1)[1].strip()
            elif ln.startswith("Validation results for run:"):
                run = ln.split("Validation results for run:", 1)[1].strip()
            elif ln.startswith("Weights:"):
                weights = ln.split("Weights:", 1)[1].strip()
            elif ln.startswith("metrics/precision(B):"):
                try: precision = float(ln.split(":", 1)[1].strip())
                except: pass
            elif ln.startswith("metrics/recall(B):"):
                try: recall = float(ln.split(":", 1)[1].strip())
                except: pass
            elif ln.startswith("metrics/mAP50(B):"):
                try: map50 = float(ln.split(":", 1)[1].strip())
                except: pass
            elif ln.startswith("metrics/mAP50-95(B):"):
                try: map5095 = float(ln.split(":", 1)[1].strip())
                except: pass
            elif ln.startswith("fitness:"):
                try: fitness = float(ln.split(":", 1)[1].strip())
                except: pass

    if not (run and weights):
        return None

    ds = infer_dataset(dataset, os.path.basename(path), weights)
    if ds not in {"base", "features"}:
        return None

    return {
        "dataset": ds,
        "run": run,
        "weights": weights,
        "mAP50": map50,
        "mAP50-95": map5095,
        "precision": precision,
        "recall": recall,
        "fitness": fitness,
        "metrics_file": os.path.basename(path),
    }

# ---- Gather rows from both patterns
rows = []
for path in sorted(glob.glob(os.path.join(TEST_RESULTS_DIR, "*_metrics.txt"))):
    rec = parse_metrics_file(path)
    if rec:
        rows.append(rec)

if not rows:
    raise SystemExit(f"No usable metrics files found in {TEST_RESULTS_DIR}")

df = pd.DataFrame(rows)

# Ensure numeric for sorting
for c in ["mAP50-95", "mAP50", "precision", "recall", "fitness"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

sort_cols = ["mAP50-95", "mAP50", "precision", "recall"]
sort_asc  = [False, False, False, False]

def save_subset(dataset_name, out_csv):
    sub = df[df["dataset"] == dataset_name].sort_values(by=sort_cols, ascending=sort_asc)
    if sub.empty:
        print(f"No rows found for dataset: {dataset_name}")
        return
    sub.to_csv(out_csv, index=False)
    top = sub.iloc[0]
    print(f"Best {dataset_name.upper()} -> run={top['run']} | mAP50-95={top['mAP50-95']:.4f} | "
          f"mAP50={top['mAP50']:.4f} | P={top['precision']:.4f} | R={top['recall']:.4f}")
    print(f"Saved: {out_csv}")

if DATASET_CHOICE in {"base", "both"}:
    save_subset("base", CSV_BASE)
if DATASET_CHOICE in {"features", "both"}:
    save_subset("features", CSV_FEAT)

In [ ]:
#@title Compares your trained models saves them into a CSV (Kangaroo)
# Compare YOLO runs with a dataset filter: "base" | "features" | "both"
# Outputs (depending on filter):
#   runs/model-results/model_comparison_base.csv
#   runs/model-results/model_comparison_features.csv

import os, glob, pandas as pd, sys

# ====== CONFIG (can override via --dataset base|features|both) ======
DATASET_CHOICE = "features"
# ====================================================================

# Optional CLI override: --dataset base | features | both
for i, a in enumerate(sys.argv):
    if a.startswith("--dataset"):
        if "=" in a:
            DATASET_CHOICE = a.split("=", 1)[1].strip().lower()
        elif i + 1 < len(sys.argv):
            DATASET_CHOICE = sys.argv[i+1].strip().lower()

if DATASET_CHOICE not in {"base", "features", "both"}:
    raise SystemExit("Invalid --dataset. Use: base | features | both")

ROOT = "/content/YoloV12-Object-Detection-Project"
TEST_RESULTS_DIR = f"{ROOT}/runs/test-results"
RESULTS_DIR = f"{ROOT}/runs/model-results"
os.makedirs(RESULTS_DIR, exist_ok=True)

CSV_BASE = os.path.join(RESULTS_DIR, "model_comparison_base.csv")
CSV_FEAT = os.path.join(RESULTS_DIR, "model_comparison_features.csv")

def infer_dataset(dataset_field, metrics_filename, weights_str):
    """Return 'base' or 'features' even if the file lacks a dataset header."""
    ds = (dataset_field or "").strip().lower()
    if ds in {"base", "features"}:
        return ds
    name = os.path.basename(metrics_filename).lower()
    w = (weights_str or "").lower()
    # Heuristics for features
    if name.startswith("features_") or "_features" in name:
        return "features"
    if "kangaroo-anatomy-features" in w:
        return "features"
    # Default fallback = base
    return "base"

def parse_metrics_file(path):
    dataset = ""
    run = ""
    weights = ""
    precision = recall = map50 = map5095 = fitness = None

    with open(path, "r") as f:
        for raw in f:
            ln = raw.strip()
            if ln.startswith("dataset:"):
                dataset = ln.split(":", 1)[1].strip()
            elif ln.startswith("Validation results for run:"):
                run = ln.split("Validation results for run:", 1)[1].strip()
            elif ln.startswith("Weights:"):
                weights = ln.split("Weights:", 1)[1].strip()
            elif ln.startswith("metrics/precision(B):"):
                try: precision = float(ln.split(":", 1)[1].strip())
                except: pass
            elif ln.startswith("metrics/recall(B):"):
                try: recall = float(ln.split(":", 1)[1].strip())
                except: pass
            elif ln.startswith("metrics/mAP50(B):"):
                try: map50 = float(ln.split(":", 1)[1].strip())
                except: pass
            elif ln.startswith("metrics/mAP50-95(B):"):
                try: map5095 = float(ln.split(":", 1)[1].strip())
                except: pass
            elif ln.startswith("fitness:"):
                try: fitness = float(ln.split(":", 1)[1].strip())
                except: pass

    if not (run and weights):
        return None

    ds = infer_dataset(dataset, os.path.basename(path), weights)
    if ds not in {"base", "features"}:
        return None

    return {
        "dataset": ds,
        "run": run,
        "weights": weights,
        "mAP50": map50,
        "mAP50-95": map5095,
        "precision": precision,
        "recall": recall,
        "fitness": fitness,
        "metrics_file": os.path.basename(path),
    }

# ---- Gather rows from both patterns
rows = []
for path in sorted(glob.glob(os.path.join(TEST_RESULTS_DIR, "*_metrics.txt"))):
    rec = parse_metrics_file(path)
    if rec:
        rows.append(rec)

if not rows:
    raise SystemExit(f"No usable metrics files found in {TEST_RESULTS_DIR}")

df = pd.DataFrame(rows)

# Ensure numeric for sorting
for c in ["mAP50-95", "mAP50", "precision", "recall", "fitness"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

sort_cols = ["mAP50-95", "mAP50", "precision", "recall"]
sort_asc  = [False, False, False, False]

def save_subset(dataset_name, out_csv):
    sub = df[df["dataset"] == dataset_name].sort_values(by=sort_cols, ascending=sort_asc)
    if sub.empty:
        print(f"No rows found for dataset: {dataset_name}")
        return
    sub.to_csv(out_csv, index=False)
    top = sub.iloc[0]
    print(f"Best {dataset_name.upper()} -> run={top['run']} | mAP50-95={top['mAP50-95']:.4f} | "
          f"mAP50={top['mAP50']:.4f} | P={top['precision']:.4f} | R={top['recall']:.4f}")
    print(f"Saved: {out_csv}")

if DATASET_CHOICE in {"base", "both"}:
    save_subset("base", CSV_BASE)
if DATASET_CHOICE in {"features", "both"}:
    save_subset("features", CSV_FEAT)

In [ ]:
#@title Choose the best model from Comparison CSV
# Outputs:
#   - copies best.pt -> /models/<dataset>_<run}_best.pt
#   - writes summary -> runs/model-results/comparison/best_model_summary_<dataset>.csv
#
# Usage:
#   - Set DATASET_CHOICE = "features" (or "base")

import os, sys, shutil, pandas as pd

# -------- CONFIG --------
DATASET_CHOICE = "features"   # "base" or "features" (CLI override supported)
# ------------------------

# Optional CLI override
for i, a in enumerate(sys.argv):
    if a.startswith("--dataset"):
        if "=" in a:
            DATASET_CHOICE = a.split("=", 1)[1].strip().lower()
        elif i + 1 < len(sys.argv):
            DATASET_CHOICE = sys.argv[i+1].strip().lower()

if DATASET_CHOICE not in {"base", "features"}:
    raise SystemExit("Invalid --dataset. Use: base | features")

ROOT        = "/content/YoloV12-Object-Detection-Project"
RESULTS_DIR = f"{ROOT}/runs/model-results"
MODELS_DIR  = f"{ROOT}/models"
SUMMARY_DIR = f"{RESULTS_DIR}/comparison"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)

# Use ONLY the dataset-specific CSV
CSV_PATH = os.path.join(RESULTS_DIR, f"model_comparison_{DATASET_CHOICE}.csv")
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"Missing CSV for dataset '{DATASET_CHOICE}'. "
        f"Expected: {CSV_PATH}\nRun your comparison script for this dataset first."
    )

print(f"Reading: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)

# Required columns (dataset column not required since CSV is dataset-specific)
required_cols = {"run", "weights", "mAP50-95", "mAP50", "precision", "recall"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV missing required columns: {missing}")

# Ensure numerics for robust sorting
for c in ["mAP50-95", "mAP50", "precision", "recall", "fitness"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(-1)

if df.empty:
    raise ValueError(f"No rows found in {CSV_PATH}")

# Sort & select best (primary: mAP50-95, then mAP50, then precision, recall)
best_row = df.sort_values(
    by=["mAP50-95", "mAP50", "precision", "recall"],
    ascending=[False, False, False, False]
).iloc[0]

run = str(best_row["run"])
src = str(best_row["weights"])

if not src or not os.path.exists(src):
    raise FileNotFoundError(f"best.pt not found at recorded path: {src}")

dst_name = f"{DATASET_CHOICE}_{run}_best.pt"
dst_path = os.path.join(MODELS_DIR, dst_name)

print(f"\nBest model ({DATASET_CHOICE.upper()}):")
print(f"  run       : {run}")
print(f"  mAP50-95  : {best_row['mAP50-95']:.4f}")
print(f"  mAP50     : {best_row['mAP50']:.4f}")
print(f"  precision : {best_row['precision']:.4f}")
print(f"  recall    : {best_row['recall']:.4f}")
print(f"Copying:\n  {src}\n  -> {dst_path}")

shutil.copy2(src, dst_path)

summary_csv = os.path.join(SUMMARY_DIR, f"best_model_summary_{DATASET_CHOICE}.csv")
best_row.to_frame().T.to_csv(summary_csv, index=False)

print(f"\nSummary written to: {summary_csv}")
print(f"Model copied to   : {dst_path}")

# Step 9 — Exporting

For step 9, users will export the best model for deployment and aslo export the respository to be saved.

## 9.1 - Export the model to deployment ready onnx file

The script below will run and export the best model chosen and convert it to an ONNX file which can be used for deployment and running on software.

In [ ]:
#@title Export our best trained model to ONNX File Format

from ultralytics import YOLO
import os, shutil

# Edit Path to your best selected model
PT_PATH = "/content/YoloV12-Object-Detection-Project/models/features_train4_best.pt" #change path manually
EXPORT_PATH = "/content/YoloV12-Object-Detection-Project/runs/final-models-exports/kangaroo_final.onnx" #change onnx file name (leave path as is)


# Ensure export directory exists
os.makedirs(os.path.dirname(EXPORT_PATH), exist_ok=True)

# Load model
model = YOLO(PT_PATH)

# Export to ONNX (Ultralytics will create it in a temp folder) (set the correct config from your models)
onnx_tmp_path = model.export(format="onnx", dynamic=True, imgsz=512)

# Move/rename to your chosen path
if os.path.exists(EXPORT_PATH):
    os.remove(EXPORT_PATH)
shutil.move(onnx_tmp_path, EXPORT_PATH)

print(f"ONNX saved to: {EXPORT_PATH}")


## 9.2 - Exporting Repository to zip file (locally and google drive)

You can use this step to export your repository to save your work or when you have completed full training

In [ ]:
#@title Export Repository as Zip File (downloads locally to host machine)
import shutil

shutil.make_archive('YoloV12-Object-Detection-Project', 'zip', '/content/YoloV12-Object-Detection-Project')

from google.colab import files
files.download('YoloV12-Object-Detection-Project.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title Export Repository as Zip File To Google Drive (downloads to google drive cloud)

from google.colab import drive
import shutil
import os

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Step 2: Define source folder (local) and destination zip path (Drive)
source_folder = '/content/YoloV12-Object-Detection-Project'  # local folder to zip
zip_name = 'YoloV12-Object-Detection-Project.zip'
zip_local_path = f'/content/{zip_name}'
drive_destination = f'/content/drive/MyDrive/{zip_name}'  # overwrite on Drive

# Step 3: Zip the source folder
if os.path.exists(zip_local_path):
    os.remove(zip_local_path)  # remove old local zip if exists
shutil.make_archive(base_name=zip_local_path.replace('.zip', ''), format='zip', root_dir=source_folder)
print(f"Zipped folder: {zip_local_path}")

# Step 4: Copy zip to Google Drive (overwrite if exists)
if os.path.exists(drive_destination):
    os.remove(drive_destination)
shutil.copy2(zip_local_path, drive_destination)
print(f"Copied {zip_local_path} to {drive_destination}")

# Step 5: Unmount Google Drive and clean up
drive.flush_and_unmount()
shutil.rmtree('/content/drive', ignore_errors=True)
print("Google Drive unmounted and cleaned up.")


# Step 10 - Troubleshooting


- **Weights YAML missing:** Ensure `models/yolo12s.pt` and `yaml/data.yaml` exist in the repo or update paths.
- **Out of Memory:** Make sure you set `batch=` as *-1* for auto batching.
- **Setting Workers:** If you are using T4 GPU set `workers=` as *4 or lower* and If you are using A100 GPU set `workers=` *as 12 or lower*
- **Val fails:** Train first; then rerun the validate cell.
- **Poor metrics:** Add more data, correct labels, tune `imgsz`/`batch`/`epochs`.
